Load the data

In [1]:
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/DataCoSupplyChainDataset.csv")

df = pd.read_csv(
    file_path,
    encoding="latin1",
    low_memory=False
)

print("Original shape:", df.shape)


Original shape: (180519, 53)


Select the columns we actually need

In [3]:
columns_to_keep = [
    "Order Id",
    "Order Item Id",
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Delivery Status",
    "Late_delivery_risk",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Shipping Mode",
    "Market",
    "Order Region",
    "Order Country",
    "Order State",
    "Order City",
    "Order Status",
    "Order Item Quantity",
    "Sales"
]

df_clean = df[columns_to_keep].copy()

print("Selected columns:", df_clean.shape[1])
df_clean.head()


Selected columns: 17


,Order Id,Order Item Id,order date (DateOrders),shipping date (DateOrders),Delivery Status,Late_delivery_risk,Days for shipping (real),Days for shipment (scheduled),Shipping Mode,Market,Order Region,Order Country,Order State,Order City,Order Status,Order Item Quantity,Sales
0,77202,180517,1/31/2018 22:56,2/3/2018 22:56,Advance shipping,0,3,4,Standard Class,Pacific Asia,Southeast Asia,Indonesia,Java Occidental,Bekasi,COMPLETE,1,327.75
1,75939,179254,1/13/2018 12:27,1/18/2018 12:27,Late delivery,1,5,4,Standard Class,Pacific Asia,South Asia,India,Rajastán,Bikaner,PENDING,1,327.75
2,75938,179253,1/13/2018 12:06,1/17/2018 12:06,Shipping on time,0,4,4,Standard Class,Pacific Asia,South Asia,India,Rajastán,Bikaner,CLOSED,1,327.75
3,75937,179252,1/13/2018 11:45,1/16/2018 11:45,Advance shipping,0,3,4,Standard Class,Pacific Asia,Oceania,Australia,Queensland,Townsville,COMPLETE,1,327.75
4,75936,179251,1/13/2018 11:24,1/15/2018 11:24,Advance shipping,0,2,4,Standard Class,Pacific Asia,Oceania,Australia,Queensland,Townsville,PENDING_PAYMENT,1,327.75


Rename columns

In [4]:
df_clean = df_clean.rename(columns={
    "Order Id": "order_id",
    "Order Item Id": "order_item_id",
    "order date (DateOrders)": "order_date",
    "shipping date (DateOrders)": "shipping_date",
    "Delivery Status": "delivery_status",
    "Late_delivery_risk": "late_delivery_risk",
    "Days for shipping (real)": "actual_shipping_days",
    "Days for shipment (scheduled)": "scheduled_shipping_days",
    "Shipping Mode": "shipping_mode",
    "Market": "market",
    "Order Region": "order_region",
    "Order Country": "order_country",
    "Order State": "order_state",
    "Order City": "order_city",
    "Order Status": "order_status",
    "Order Item Quantity": "quantity",
    "Sales": "sales"
})

df_clean.head()


,order_id,order_item_id,order_date,shipping_date,delivery_status,late_delivery_risk,actual_shipping_days,scheduled_shipping_days,shipping_mode,market,order_region,order_country,order_state,order_city,order_status,quantity,sales
0,77202,180517,1/31/2018 22:56,2/3/2018 22:56,Advance shipping,0,3,4,Standard Class,Pacific Asia,Southeast Asia,Indonesia,Java Occidental,Bekasi,COMPLETE,1,327.75
1,75939,179254,1/13/2018 12:27,1/18/2018 12:27,Late delivery,1,5,4,Standard Class,Pacific Asia,South Asia,India,Rajastán,Bikaner,PENDING,1,327.75
2,75938,179253,1/13/2018 12:06,1/17/2018 12:06,Shipping on time,0,4,4,Standard Class,Pacific Asia,South Asia,India,Rajastán,Bikaner,CLOSED,1,327.75
3,75937,179252,1/13/2018 11:45,1/16/2018 11:45,Advance shipping,0,3,4,Standard Class,Pacific Asia,Oceania,Australia,Queensland,Townsville,COMPLETE,1,327.75
4,75936,179251,1/13/2018 11:24,1/15/2018 11:24,Advance shipping,0,2,4,Standard Class,Pacific Asia,Oceania,Australia,Queensland,Townsville,PENDING_PAYMENT,1,327.75


Convert dates

In [5]:
df_clean["order_date"] = pd.to_datetime(
    df_clean["order_date"],
    format="%m/%d/%Y %H:%M",
    errors="coerce"
)

df_clean["shipping_date"] = pd.to_datetime(
    df_clean["shipping_date"],
    format="%m/%d/%Y %H:%M",
    errors="coerce"
)

print(df_clean[["order_date", "shipping_date"]].dtypes)


order_date       datetime64[ns]
shipping_date    datetime64[ns]
dtype: object


Check data quality

In [6]:
print("Duplicate order item IDs:",
      df_clean["order_item_id"].duplicated().sum())

print("Missing values:")
print(df_clean.isnull().sum())

print("Unique orders:", df_clean["order_id"].nunique())

print("Unique order items:", df_clean["order_item_id"].nunique())


Duplicate order item IDs: 0
Missing values:
order_id                   0
order_item_id              0
order_date                 0
shipping_date              0
delivery_status            0
late_delivery_risk         0
actual_shipping_days       0
scheduled_shipping_days    0
shipping_mode              0
market                     0
order_region               0
order_country              0
order_state                0
order_city                 0
order_status               0
quantity                   0
sales                      0
dtype: int64
Unique orders: 65752
Unique order items: 180519


Validate the records

In [7]:
invalid_records = df_clean[
    df_clean["order_id"].isna()
    | df_clean["order_item_id"].isna()
    | df_clean["order_date"].isna()
    | df_clean["shipping_date"].isna()
    | (df_clean["actual_shipping_days"] < 0)
    | (df_clean["scheduled_shipping_days"] < 0)
]

print("Invalid records:", len(invalid_records))


Invalid records: 0


Save the cleaned dataset

In [8]:
output_path = Path("../data/processed")
output_path.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    output_path / "dataco_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully!")
print("Final shape:", df_clean.shape)

Cleaned dataset saved successfully!
Final shape: (180519, 17)
